# MDI results preserved from the local analysis

This notebook preserves recorded outputs for the missing-data imputation (MDI)
results discussed with the INTSYS 2024 pipeline. It is a **result-extract**, not
the full MDI workflow: the code cells below rely on helpers and restricted data
from a separate local analysis and are kept for their recorded outputs.

The `2_missing_emission_data_imputation.ipynb` notebook here is a cleaned step-2
derivative and is not a complete record of every MDI experiment.


## XGBoost with embeddings

Here the model uses graph embeddings as extra features to predict emissions on road links.


In [ ]:
# Predict with XGBoost using the embedding features.
model = xgb.XGBRegressor(learning_rate = 0.05, n_estimators = 500, max_depth = 6, eval_metric='mape')
prediction_errors = predict_road_emissions(embs_n_co2m, model, ratio=0.9, shuffles=10, messages=False)

Time: 355 seconds
Mean MAPE is 185.17\% (11.60\%), mean MAE is 113.66%


## Hybrid classification–regression result

This two-step approach first labels roads as low or high emitters, then predicts a value within each group.


In [ ]:
mapes_train = list()
maes_train = list()

mapes_test = list()
maes_test = list()

# Roads classified as high emitters.
predicted_high_emission_roads = embs_n_co2m[embs_n_co2m['prediction']==1]

for s in range(10):
    regressor=xgb.XGBRegressor(learning_rate = 0.12, n_estimators = 550,
                               max_depth = 7, eval_metric = 'mape')

    # Fit on measured high-emission roads.
    X, y = high_emission_roads.drop(['road_link', 'start_point', 'end_point', 'CO_2/m', 'emission_class'], axis=1), high_emission_roads['CO_2/m']
    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, test_size=0.1)
    y_train_log = np.log(y_train)
    y_test_log = np.log(y_test)

    regressor.fit(X_train, y_train_log)
    preds = np.exp(regressor.predict(X_test))

    mape_train = metrics.mean_absolute_percentage_error(y_test, preds)
    mae_train = metrics.mean_absolute_error(y_test, preds)
    mapes_train.append(mape_train)
    maes_train.append(mae_train)

    # Evaluate on roads classified as high emitters.
    X, y = predicted_high_emission_roads.drop(['road_link', 'start_point', 'end_point', 'CO_2/m', 'emission_class', 'prediction'], axis=1), predicted_high_emission_roads['CO_2/m']
    y_log = np.log(y)

    preds = np.exp(regressor.predict(X))

    mape_test = metrics.mean_absolute_percentage_error(y, preds)
    mae_test = metrics.mean_absolute_error(y, preds)
    mapes_test.append(mape_test)
    maes_test.append(mae_test)

print("Mean MAPE (on real values) is %.2f%% (std is %.3f), mean MAE is %.2f%%" %(np.mean(mapes_train)*100, np.std(mapes_train), np.mean(maes_train)*100))
print("Mean MAPE (on classified values) is %.2f%% (std is %.3f), mean MAE is %.2f%%" %(np.mean(mapes_test)*100, np.std(mapes_test), np.mean(maes_test)*100))

Mean MAPE (on real values) is 82.08% (std is 0.018), mean MAE is 290.85%
Mean MAPE (on classified values) is 94.53% (std is 0.009), mean MAE is 100.93%


In [ ]:
errors_all = low_emission_errors.append(high_emission_errors, ignore_index=True)
print("MAPE across all roads is %.2f%% (std %.2f%%)\n" %(np.mean(errors_all.percent_error), np.std(errors_all.percent_error)))
errors_all.head()


MAPE across all roads is 124.82% (std 241.14%)



,road_link,start_point,end_point,CO_2/m,prediction,difference,percent_error
0,"(2480536523, 10696669080, 0)",2480536523,10696669080,2.605964,0.866490,1.739474,66.749733
1,"(3752747941, 1675060943, 0)",3752747941,1675060943,0.620309,0.386222,0.234087,37.737185
2,"(280714901, 280714900, 0)",280714901,280714900,0.011947,0.109778,0.097831,818.841838
3,"(2741074035, 2741074034, 0)",2741074035,2741074034,0.095460,0.033549,0.061911,64.855572
4,"(2688307578, 10808019462, 0)",2688307578,10808019462,0.177517,0.063977,0.113540,63.960191


In [ ]:
city = 'Pisa'
path = '/content/drive/MyDrive/Colab Notebooks/phd_data/xgb_xgb/'
date = '06_21_2024'

errors_low = pd.read_csv(path+'low_emission_log_errors_{}_{}_mape_124.csv'.format(city.lower(), date))
errors_high = pd.read_csv(path+'high_emission_log_errors_{}_{}_mape_136.csv'.format(city.lower(), date))
errors_all = errors_low.append(errors_high, ignore_index=True)
print(np.mean(errors_low.percent_error), np.std(errors_low.percent_error))
print(np.mean(errors_high.percent_error), np.std(errors_high.percent_error))
print(np.mean(errors_all.percent_error), np.std(errors_all.percent_error))
errors_all.info()


123.79903448722364 243.45732344765452
136.01877093829583 213.69127564262365
124.81684693047808 241.1420047743598
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2041 entries, 0 to 2040
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   road_link      2041 non-null   object 
 1   start_point    2041 non-null   int64  
 2   end_point      2041 non-null   int64  
 3   CO_2/m         2041 non-null   float64
 4   prediction     2041 non-null   float64
 5   difference     2041 non-null   float64
 6   percent_error  2041 non-null   float64
dtypes: float64(4), int64(2), object(1)
memory usage: 111.7+ KB


## Analysis by road length

The error is broken down by road length to see whether short or long roads are harder to predict.


In [ ]:
for key in emissions.keys():
  print("Here %s-emission roads are divided into 5 quantiles by lenght.\nBelow are MAPE results for each quantile:\n" %key)

  for quantile, n in enumerate([x/10 for x in range(0, 10, 2)]):
    errors = emissions[key]
    mape_quantile = errors[(errors.length > errors.length.quantile(n)) & (errors.length <= errors.length.quantile(n+0.2))]
    print("Quantile %s (%.0f-%.0f percent): mean %.2f\%% (%.2f\%%)"\
          %(quantile+1, n*100, n*100+20, np.mean(mape_quantile.percent_error), np.std(mape_quantile.percent_error)))
  print()

Here low-emission roads are divided into 5 quantiles by lenght.
Below are MAPE results for each quantile:

Quantile 1 (0-20 percent): mean 102.35\% (132.20\%)
Quantile 2 (20-40 percent): mean 110.29\% (181.97\%)
Quantile 3 (40-60 percent): mean 117.28\% (170.33\%)
Quantile 4 (60-80 percent): mean 125.40\% (256.94\%)
Quantile 5 (80-100 percent): mean 163.89\% (385.46\%)

Here high-emission roads are divided into 5 quantiles by lenght.
Below are MAPE results for each quantile:

Quantile 1 (0-20 percent): mean 227.74\% (303.86\%)
Quantile 2 (20-40 percent): mean 122.18\% (171.38\%)
Quantile 3 (40-60 percent): mean 174.71\% (231.43\%)
Quantile 4 (60-80 percent): mean 114.28\% (186.58\%)
Quantile 5 (80-100 percent): mean 42.54\% (42.86\%)

Here all-emission roads are divided into 5 quantiles by lenght.
Below are MAPE results for each quantile:

Quantile 1 (0-20 percent): mean 103.33\% (131.64\%)
Quantile 2 (20-40 percent): mean 126.65\% (210.57\%)
Quantile 3 (40-60 percent): mean 111.54\% (

## Analysis by speed limit

The error is broken down by speed limit, which acts as a proxy for road type and traffic conditions.


In [ ]:
for key in emissions.keys():
  print("Distribution of maxspeed values\n(%s emission roads):\n" %key)
  errors = emissions[key]
  for speed in errors.maxspeed.unique():
    mape_speed = errors[errors['maxspeed'] == speed]
    mean_speed = np.mean(mape_speed.percent_error)
    std_speed = np.std(mape_speed.percent_error)

    print("There are %s roads with %s km/h maxspeed, the mean MAPE is %.2f\%% (%.2f\%%)" %(len(mape_speed), speed, mean_speed, std_speed))
  print()

Distribution of maxspeed values
(low emission roads):

There are 1393 roads with NG km/h maxspeed, the mean MAPE is 123.33\% (206.90\%)
There are 239 roads with 50 km/h maxspeed, the mean MAPE is 98.61\% (149.56\%)
There are 11 roads with 10 km/h maxspeed, the mean MAPE is 287.67\% (400.96\%)
There are 36 roads with 40 km/h maxspeed, the mean MAPE is 97.95\% (75.52\%)
There are 151 roads with 30 km/h maxspeed, the mean MAPE is 156.54\% (513.14\%)
There are 25 roads with 90 km/h maxspeed, the mean MAPE is 151.09\% (347.67\%)
There are 3 roads with 70 km/h maxspeed, the mean MAPE is 69.57\% (12.71\%)
There are 10 roads with 60 km/h maxspeed, the mean MAPE is 87.72\% (74.18\%)
There are 3 roads with 20 km/h maxspeed, the mean MAPE is 354.91\% (302.93\%)

Distribution of maxspeed values
(high emission roads):

There are 23 roads with 40 km/h maxspeed, the mean MAPE is 122.31\% (104.93\%)
There are 42 roads with NG km/h maxspeed, the mean MAPE is 226.69\% (281.93\%)
There are 37 roads with 

## Notes

These cells preserve recorded MDI outputs from a separate local analysis.
The INTSYS paper remains the authoritative source for the reported result
tables. The step-2 notebook here is a cleaned derivative and does not reproduce
every local experiment.
